# Local x402 simulation (Omnipay)

Local-only. No AWS, no live funds, no Bedrock. Synthetic proofs only.
`value_transferred=false`.

In [ ]:
from datetime import UTC, datetime, timedelta
from decimal import Decimal

from omnipay_x402 import VALUE_TRANSFERRED, live_agentcore_status
from omnipay_x402.demo import build_demo
from omnipay_x402.merchant import RESOURCE_URL
from omnipay_x402.models import ApprovalGrant, PurchaseRequest

assert VALUE_TRANSFERRED is False
print(live_agentcore_status())

now = datetime.now(UTC)
app, merchant, payments = build_demo(now)
purchase = PurchaseRequest(
    request_id="request-001",
    resource_url=RESOURCE_URL,
    purpose="supplier_due_diligence",
    idempotency_key="purchase-001",
)
grant = ApprovalGrant(
    approval_id="approval-001",
    request_id=purchase.request_id,
    resource_url=purchase.resource_url,
    purpose=purchase.purpose,
    maximum_amount=Decimal("0.25"),
    approved_by="synthetic-reviewer",
    approved_at=now,
    expires_at=now + timedelta(minutes=10),
)
result = app.purchase(purchase, approval=grant, now=now)
print(result.status, result.receipt.receipt_id)
print([e.event_type for e in result.audit_events])